### 1 — Import YOLO

In [1]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

## 2 — Model path

In [2]:
MODEL_PATH = "../best.pt"

model = YOLO(MODEL_PATH)

print("Model loaded successfully!")
print("Classes:", model.names)

Model loaded successfully!
Classes: {0: 'D00', 1: 'D10', 2: 'D20', 3: 'D40', 4: 'D43'}


## 3 — Dataset YAML load karein

In [3]:
DATA_YAML = "../dataset.yaml"

print("Dataset YAML:", DATA_YAML)

Dataset YAML: ../dataset.yaml


## 4 — Test Dataset Evaluation

In [5]:
import os

print(os.getcwd())

d:\Deep Learning\Road-Damage-Detection\notebooks


In [6]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("dataset.yaml exists:", Path("../dataset.yaml").exists())
print("Root dataset.yaml exists:", Path("dataset.yaml").exists())

Current folder: d:\Deep Learning\Road-Damage-Detection\notebooks
dataset.yaml exists: False
Root dataset.yaml exists: False


In [8]:
DATA_YAML = r"D:\Deep Learning\Road-Damage-Detection\dataset.yaml"

print(Path(DATA_YAML).exists())

True


In [9]:
metrics = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16
)

Ultralytics 8.4.112  Python-3.11.0 torch-2.13.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)
WARNING val: Slow image access detected (ping: 0.50.1 ms, read: 8.22.8 MB/s, size: 74.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\Deep Learning\Road-Damage-Detection\dataset\RDD_SPLIT\test\labels... 5758 images, 1790 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5758/5758 234.1it/s 24.6s0.1ss
val: New cache created: D:\Deep Learning\Road-Damage-Detection\dataset\RDD_SPLIT\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 360/360 2.1s/it 12:202.1ss
                   all       5758       9675      0.608      0.547       0.57      0.304
                   D00       2080       3925      0.601      0.519      0.539      0.297
                   D10       1118       1675      0.556      0.521      0.515      

In [10]:
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Precision: 0.6084901349557382
Recall: 0.5474183591367116
mAP50: 0.5699243280342645
mAP50-95: 0.3044182859334288


## 5 — Main Metrics

In [11]:
print("Evaluation Results")
print("=" * 40)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Evaluation Results
mAP50: 0.5699243280342645
mAP50-95: 0.3044182859334288


## 6 — Precision & Recall

In [12]:
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Precision: 0.6084901349557382
Recall: 0.5474183591367116


## 7 — Per-Class Results

In [13]:
print("Per-Class Results")
print("=" * 40)

for class_id, class_name in model.names.items():

    try:
        print(
            class_id,
            class_name,
            "mAP50:",
            metrics.box.ap50[class_id]
        )

    except Exception:
        pass

Per-Class Results
0 D00 mAP50: 0.539179858782403
1 D10 mAP50: 0.5153249870545511
2 D20 mAP50: 0.6418146902607391
3 D40 mAP50: 0.7177655883661385
4 D43 mAP50: 0.43553651570749086


## 8 — Metrics Table

In [15]:
import pandas as pd

evaluation_results = {
    "Metric": [
        "Precision",
        "Recall",
        "mAP50",
        "mAP50-95"
    ],
    "Score": [
        metrics.box.mp,
        metrics.box.mr,
        metrics.box.map50,
        metrics.box.map
    ]
}

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,Metric,Score
0,Precision,0.608490
1,Recall,0.547418
2,mAP50,0.569924
3,mAP50-95,0.304418


## 9 — Metric Graph

In [16]:
plt.figure(figsize=(8, 5))

plt.bar(
    evaluation_df["Metric"],
    evaluation_df["Score"]
)

plt.title("Model Evaluation Metrics")
plt.xlabel("Metric")
plt.ylabel("Score")

plt.ylim(0, 1)

plt.show()

<Figure size 800x500 with 1 Axes>

## 10 — Test Image Prediction

In [17]:
TEST_IMAGE = "../test_images/China_Drone_000008.jpg"

results = model.predict(
    source=TEST_IMAGE,
    imgsz=640,
    conf=0.25,
    save=True
)


image 1/1 d:\Deep Learning\Road-Damage-Detection\notebooks\..\test_images\China_Drone_000008.jpg: 640x640 1 D10, 127.5ms
Speed: 7.1ms preprocess, 127.5ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 640)
Results saved to D:\Deep Learning\Road-Damage-Detection\notebooks\runs\detect\predict


## 11 — Prediction Display

In [18]:
result = results[0]

annotated_image = result.plot()

plt.figure(figsize=(12, 8))
plt.imshow(annotated_image)
plt.axis("off")
plt.title("Model Prediction")
plt.show()

<Figure size 1200x800 with 1 Axes>

## 12 — Prediction Details

In [19]:
if result.boxes is not None and len(result.boxes) > 0:

    for box in result.boxes:

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        print(
            f"Class: {model.names[class_id]}"
        )

        print(
            f"Confidence: {confidence:.2%}"
        )

else:

    print("No damage detected.")

Class: D10
Confidence: 52.66%


## 13 — Multiple Test Images

In [20]:
test_images = list(
    Path("../test_images").glob("*.jpg")
)

print("Test images found:", len(test_images))

Test images found: 1


In [21]:
test_images = []

for extension in ["*.jpg", "*.jpeg", "*.png"]:
    test_images.extend(
        Path("../test_images").glob(extension)
    )

print("Test images found:", len(test_images))

Test images found: 1


## 14 — Sample Predictions

In [22]:
sample_test_images = test_images[:6]

for image_path in sample_test_images:

    results = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        verbose=False
    )

    result = results[0]

    annotated_image = result.plot()

    plt.figure(figsize=(10, 6))
    plt.imshow(annotated_image)
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()

<Figure size 1000x600 with 1 Axes>

## 15 Model Evaluation Summary

The trained YOLO road damage detection model was evaluated using the test dataset.

The evaluation included:

* Precision
* Recall
* mAP50
* mAP50-95
* Per-class detection performance
* Sample prediction visualization

The model was also tested on sample road images to verify that it can identify road damage and display confidence scores with bounding boxes.

The evaluation results provide an overall assessment of the trained model before deployment in the Streamlit application.
